# Day 9: FASTQ と read quality control

## この章で解く現実の課題

シーケンサーから得られたFASTQ readは、発現解析に使ってよい品質なのかを判断します。

count matrixの前にはFASTQがあります。FASTQは、読まれた塩基配列と、その塩基ごとの品質スコアを持つファイルです。品質が低いreadやadapter混入が多いと、アライメントや定量に影響します。


## キーワード辞典（この章）

| キーワード | まずこう理解する | この章での使いどころ | よくある誤解 |
|---|---|---|---|
| FASTQ | read配列+品質を持つ生データ形式 | count matrix前段の理解 | count matrixが唯一の入力と思う |
| Phred score | 塩基読み取り品質の数値 | 低品質領域の判断 | 値の意味（誤り確率）を見ない |
| adapter | ライブラリ由来の余分配列 | trimming判断の対象 | 無視しても定量に影響しないと思う |
| per-base quality | 塩基位置ごとの品質分布 | read末端品質低下の把握 | 平均品質だけ見れば十分と思う |
| duplication | 同一readの重複度 | PCR偏りの手がかり | 重複が高い=必ず失敗と断定する |
| GC content | GC割合分布 | 汚染やライブラリ偏りの確認 | 期待分布は常に同じと思う |



## この章での判断軸

1. 低品質・adapter混入の有無をまず確認する。
2. 改善手段（trim/filter）の副作用も考える。
3. QC判断を記録し、後段解析の解釈とつなげる。


## FASTQの4行構造

1 readは通常4行で表されます。

1. `@read_id`: readの名前
2. `ACGT...`: 塩基配列
3. `+`: 区切り行
4. `IIII...`: 各塩基の品質スコア

品質文字はPhred scoreに変換されます。ざっくり言うと、スコアが高いほどその塩基が正しく読まれた可能性が高いです。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fastq_record = [
    "@read_001",
    "ACGTTGCAACGT",
    "+",
    "IIIIIII####I",
]

pd.DataFrame({"line": [1, 2, 3, 4], "content": fastq_record, "meaning": ["read ID", "sequence", "separator", "quality"]})


In [ ]:
def phred33_to_q(char):
    # FASTQでよく使われるPhred+33形式。
    # 文字をASCII番号にし、33を引くと品質スコアになる。
    return ord(char) - 33

quality_string = fastq_record[3]
qualities = [phred33_to_q(c) for c in quality_string]

quality_df = pd.DataFrame({
    "base_position": range(1, len(qualities) + 1),
    "base": list(fastq_record[1]),
    "quality_char": list(quality_string),
    "phred_score": qualities,
})
quality_df


In [ ]:
ax = quality_df.plot(x="base_position", y="phred_score", marker="o", legend=False, figsize=(7, 3))
ax.axhline(20, color="red", linestyle="--", label="Q20")
ax.set_ylabel("Phred quality score")
ax.set_title("Per-base quality for one example read")
ax.legend()
plt.tight_layout()
plt.show()


## 読み取り

Phred score 20は、ざっくり1%程度の誤り確率に対応します。実際のFastQCでは、1 readではなく大量のreadを位置ごとに集計して、read後半で品質が落ちていないか、adapter配列が残っていないかを見ます。

RNA-seq解析では、低品質readが多い場合、trimやfilterを検討します。ただし、何でも削れば良いわけではありません。削りすぎるとreadが短くなり、どの遺伝子由来か分かりにくくなることがあります。


In [ ]:
qc_summary = pd.DataFrame({
    "QC_item": ["per base quality", "adapter content", "sequence duplication", "GC content"],
    "bad_pattern": [
        "read後半でQ20未満が多い",
        "adapter配列が高頻度に検出される",
        "同じreadが異常に多い",
        "期待分布から大きく外れる",
    ],
    "possible_action": [
        "末端trimを検討",
        "adapter trimmingを実施",
        "PCR重複や低多様性を疑う",
        "汚染、細胞種、ライブラリタイプを確認",
    ],
})
qc_summary


## エラーが出た場合

- `ord() expected a character`: 品質文字列を1文字ずつ処理できていない可能性があります。
- グラフが空: `quality_df` を作るセルを先に実行してください。

## この章のゴール

FASTQの4行が何を表し、品質スコアが低いと後段解析にどう影響するか説明できれば合格です。
